new


In [ ]:
!pip install ddgs sentence-transformers transformers accelerate bitsandbytes

In [ ]:
import re
import torch
from ddgs import DDGS
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
class ClaimExtractor:

    def extract_claims(self, article):

        sentences = re.split(r'[.!?]+', article)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 25]

        claims = sentences[:3]

        return claims

In [ ]:
class QueryGenerator:

    def generate_queries(self, claims):

        queries = []

        for claim in claims:

            words = re.findall(r'\b[A-Za-z0-9]+\b', claim)

            keywords = [w for w in words if len(w) > 3]

            queries.append(" ".join(keywords[:8]))

        return queries[:5]

In [ ]:
class KnowledgeRetriever:

    def __init__(self):

        self.credible_domains = {
            "reuters.com":1.0,
            "bbc.com":1.0,
            "apnews.com":1.0,
            "nytimes.com":0.95,
            "theguardian.com":0.95,
            "washingtonpost.com":0.95,
            "nature.com":0.9,
            "sciencedaily.com":0.85
        }

    def search(self, query):

        evidence = []

        with DDGS() as ddgs:

            results = ddgs.text(query, max_results=10)

            for r in results:

                url = r["href"]

                credibility = 0.6

                for domain,score in self.credible_domains.items():
                    if domain in url:
                        credibility = score

                evidence.append({
                    "title": r["title"],
                    "snippet": r["body"],
                    "url": url,
                    "credibility_score": credibility
                })

        return evidence

In [ ]:
class EvidenceRanker:

    def __init__(self):

        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')

    def rank(self, claim, evidence):

        if not evidence:
            return []

        claim_embedding = self.encoder.encode(claim, convert_to_tensor=True)

        snippets = [e["snippet"] for e in evidence]

        snippet_embeddings = self.encoder.encode(snippets, convert_to_tensor=True)

        similarities = util.cos_sim(claim_embedding, snippet_embeddings)[0]

        for i,e in enumerate(evidence):

            semantic_score = float(similarities[i])

            e["score"] = 0.7*semantic_score + 0.3*e["credibility_score"]

        ranked = sorted(evidence, key=lambda x:x["score"], reverse=True)

        return ranked[:5]

In [ ]:
class ClaimVerifier:

    def __init__(self):

        model_name = "mistralai/Mistral-7B-Instruct-v0.2"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto"
        )

    def verify(self, claim, evidence):

        evidence_text = "\n".join([e["snippet"] for e in evidence[:3]])

        prompt = f"""
You are a fact-checking assistant.

Claim:
{claim}

Evidence:
{evidence_text}

Determine if the claim is TRUE or FALSE.

Respond exactly like this:

VERDICT: TRUE or FALSE
CONFIDENCE: number between 0 and 1
EXPLANATION: short explanation
"""

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():

            outputs = self.model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.0
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        verdict = "REAL"
        confidence = 0.5
        explanation = ""

        lines = response.split("\n")

        for line in lines:

            if "VERDICT" in line.upper():

                if "FALSE" in line.upper():
                    verdict = "FAKE"
                else:
                    verdict = "REAL"

            if "CONFIDENCE" in line.upper():

                try:
                    confidence = float(line.split(":")[-1].strip())
                except:
                    pass

            if "EXPLANATION" in line.upper():

                explanation = line.split(":",1)[1].strip()

        return verdict, confidence, explanation

In [ ]:
class OutputGenerator:

    def display(self, claim, verdict, confidence, explanation, evidence):

        print("\n==============================")
        print("FAKE NEWS DETECTION RESULT")
        print("==============================")

        print("\nClaim:")
        print(claim)

        print("\nFinal Verdict:", verdict)

        print("Confidence:", round(confidence,2))

        print("\nExplanation:")
        print(explanation)

        print("\nTop Evidence Sources:")

        for i,e in enumerate(evidence[:3],1):

            print(f"{i}. {e['title']}")
            print(f"   {e['url']}\n")

        print("==============================")

In [ ]:
class FakeNewsPipeline:

    def __init__(self):

        self.extractor = ClaimExtractor()
        self.query_gen = QueryGenerator()
        self.retriever = KnowledgeRetriever()
        self.ranker = EvidenceRanker()
        self.verifier = ClaimVerifier()
        self.output = OutputGenerator()

    def detect(self, article):

        claims = self.extractor.extract_claims(article)

        queries = self.query_gen.generate_queries(claims)

        evidence = []

        for q in queries:

            evidence.extend(self.retriever.search(q))

        ranked_evidence = self.ranker.rank(claims[0], evidence)

        verdict, confidence, explanation = self.verifier.verify(
            claims[0],
            ranked_evidence
        )

        self.output.display(
            claims[0],
            verdict,
            confidence,
            explanation,
            ranked_evidence
        )

In [ ]:
pipeline = FakeNewsPipeline()

article = """
Scientists claim drinking 10 cups of coffee daily can extend lifespan by 50 years.
"""

pipeline.detect(article)

# Evaluation

In [ ]:
!pip install scikit-learn

In [ ]:
test_data = [

{
"article": "Scientists claim drinking 10 cups of coffee daily can extend lifespan by 50 years.",
"label": "FAKE"
},

{
"article": "NASA confirmed the presence of water molecules on the moon in 2020.",
"label": "REAL"
},

{
"article": "COVID-19 vaccines cause infertility in women according to scientists.",
"label": "FAKE"
},

{
"article": "The James Webb Space Telescope captured the deepest infrared image of the universe.",
"label": "REAL"
},

{
"article": "A viral post claims that eating garlic cures all types of cancer instantly.",
"label": "FAKE"
}

]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true = []
y_pred = []

for item in test_data:

    article = item["article"]
    true_label = item["label"]

    claims = pipeline.extractor.extract_claims(article)

    queries = pipeline.query_gen.generate_queries(claims)

    evidence = []

    for q in queries:
        evidence.extend(pipeline.retriever.search(q))

    ranked = pipeline.ranker.rank(claims[0], evidence)

    verdict, confidence, explanation = pipeline.verifier.verify(claims[0], ranked)

    y_true.append(true_label)
    y_pred.append(verdict)

    print("\nArticle:", article[:80])
    print("True:", true_label)
    print("Predicted:", verdict)

In [ ]:
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred, pos_label="REAL")

recall = recall_score(y_true, y_pred, pos_label="REAL")

f1 = f1_score(y_true, y_pred, pos_label="REAL")

print("\n==============================")
print("MODEL EVALUATION")
print("==============================")

print("Accuracy:", round(accuracy,2))
print("Precision:", round(precision,2))
print("Recall:", round(recall,2))
print("F1 Score:", round(f1,2))

# isot dataset

In [1]:
!pip install ddgs sentence-transformers transformers accelerate bitsandbytes scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 46.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01


In [10]:
import re
import torch
import pandas as pd
from ddgs import DDGS
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM,AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

############################################################
# CLAIM EXTRACTION
############################################################

class ClaimExtractor:

    def extract_claims(self, article):

        article = re.sub(r"\s+", " ", article)

        sentences = re.split(r'(?<=[.!?]) +', article)

        sentences = [s.strip() for s in sentences if len(s.strip()) > 30]

        if len(sentences) == 0:
            return [article[:200]]

        return sentences[:5]


############################################################
# QUERY GENERATION
############################################################

class QueryGenerator:

    def generate_queries(self, claims):

        queries=[]

        stopwords=set([
            "the","is","are","was","were","this","that","with","from",
            "have","has","had","they","their","about","there","which"
        ])

        for claim in claims:

            words=re.findall(r"\b[a-zA-Z0-9]+\b",claim.lower())

            keywords=[w for w in words if len(w)>3 and w not in stopwords]

            query=" ".join(keywords[:8])

            if query.strip()=="":
                query=claim[:100]

            queries.append(query)

        queries=list(set(queries))

        return queries[:5]


############################################################
# KNOWLEDGE RETRIEVAL
############################################################

class KnowledgeRetriever:

    def __init__(self):

        self.credible_domains={

            "reuters.com":1.0,
            "apnews.com":1.0,
            "bbc.com":1.0,
            "nytimes.com":0.95,
            "washingtonpost.com":0.95,
            "theguardian.com":0.95,
            "npr.org":0.95,
            "cbsnews.com":0.9,
            "abcnews.go.com":0.9,
            "nbcnews.com":0.9,
            "cnn.com":0.9,
            "usatoday.com":0.9,

            "nature.com":0.9,
            "science.org":0.9,
            "sciencedaily.com":0.9,

            "who.int":0.9,
            "cdc.gov":0.9,
            "nih.gov":0.9,

            "snopes.com":0.9,
            "factcheck.org":0.9,
            "politifact.com":0.9
        }


    def search(self, query):

        evidence=[]

        if query.strip()=="":
            return evidence

        with DDGS() as ddgs:

            results=ddgs.text(query,max_results=10)

            for r in results:

                url=r["href"]

                credibility=0.3

                for domain,score in self.credible_domains.items():

                    if domain in url:
                        credibility=score

                if credibility<0.5:
                    continue

                evidence.append({

                    "title":r["title"],
                    "snippet":r["body"],
                    "url":url,
                    "credibility_score":credibility
                })

        return evidence


############################################################
# EVIDENCE RANKING
############################################################

class EvidenceRanker:

    def __init__(self):

        self.encoder=SentenceTransformer("all-MiniLM-L6-v2")


    def rank(self, claim, evidence):

        if len(evidence)==0:
            return []

        claim_emb=self.encoder.encode(claim,convert_to_tensor=True)

        snippets=[e["snippet"] for e in evidence]

        snippet_emb=self.encoder.encode(snippets,convert_to_tensor=True)

        similarities=util.cos_sim(claim_emb,snippet_emb)[0]

        claim_words=set(claim.lower().split())

        for i,e in enumerate(evidence):

            semantic=float(similarities[i])

            snippet_words=set(e["snippet"].lower().split())

            overlap=len(claim_words & snippet_words)/(len(claim_words)+1)

            e["score"]=0.6*semantic + 0.25*e["credibility_score"] + 0.15*overlap

        ranked=sorted(evidence,key=lambda x:x["score"],reverse=True)

        return ranked[:5]


############################################################

############################################################
# CLAIM VERIFIER (HYBRID: RULE + LLM)
############################################################

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re


class ClaimVerifier:

    def __init__(self):

        model_name = "microsoft/phi-2"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto"
        )


    ########################################################
    # Rule-based contradiction detection
    ########################################################

    def rule_check(self, claim, evidence):

        text = " ".join([e["snippet"] for e in evidence]).lower()

        claim_lower = claim.lower()

        claim_numbers = re.findall(r"\d+", claim_lower)

        evidence_numbers = re.findall(r"\d+", text)

        # numeric contradiction
        if claim_numbers:
            if not any(n in evidence_numbers for n in claim_numbers):
                return "FAKE", 0.75, "Evidence contradicts numeric claim"

        # exaggerated language
        exaggeration = ["always", "never", "100%", "all", "guaranteed"]

        if any(x in claim_lower for x in exaggeration):
            return "FAKE", 0.7, "Claim uses exaggerated language"

        # miracle cure detection
        if "cure" in claim_lower and any(w in text for w in ["may", "might", "could"]):
            return "FAKE", 0.8, "Evidence contradicts strong cure claim"

        return None


    ########################################################
    # LLM verification
    ########################################################

    def llm_verify(self, claim, evidence):

        evidence_text = "\n".join([e["snippet"] for e in evidence[:3]])

        prompt = f"""
You are a professional fact-checker.

Claim:
{claim}

Evidence:
{evidence_text}

Task:
Determine if the claim is TRUE or FALSE.

Rules:
- If evidence contradicts the claim → FALSE
- If evidence supports the claim → TRUE
- If evidence is unclear → TRUE

Respond exactly in this format:

VERDICT: TRUE or FALSE
CONFIDENCE: number between 0 and 1
EXPLANATION: short explanation
"""

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(self.model.device)

        with torch.no_grad():

            outputs = self.model.generate(
                **inputs,
                max_new_tokens=120
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        verdict = "REAL"
        confidence = 0.6
        explanation = ""

        for line in response.split("\n"):

            if "VERDICT" in line.upper():

                if "FALSE" in line.upper():
                    verdict = "FAKE"

                if "TRUE" in line.upper():
                    verdict = "REAL"

            if "CONFIDENCE" in line.upper():

                nums = re.findall(r"\d*\.?\d+", line)

                if len(nums) > 0:
                    confidence = float(nums[0])

            if "EXPLANATION" in line.upper():

                parts = line.split(":", 1)

                if len(parts) > 1:
                    explanation = parts[1].strip()

        return verdict, confidence, explanation


    ########################################################
    # Main verification function
    ########################################################

    def verify(self, claim, evidence):

        if len(evidence) == 0:
            return "FAKE", 0.3, "No credible evidence retrieved"

        # rule-based check first
        rule_result = self.rule_check(claim, evidence)

        if rule_result:
            return rule_result

        # otherwise use LLM
        return self.llm_verify(claim, evidence)


############################################################
# PIPELINE
############################################################

class FakeNewsPipeline:

    def __init__(self):

        self.extractor=ClaimExtractor()
        self.query_gen=QueryGenerator()
        self.retriever=KnowledgeRetriever()
        self.ranker=EvidenceRanker()
        self.verifier=ClaimVerifier()


    def detect(self,article):

        claims=self.extractor.extract_claims(article)

        queries=self.query_gen.generate_queries(claims)

        evidence=[]

        for q in queries:
            evidence.extend(self.retriever.search(q))

        if len(evidence)==0:
            return {"verdict":"FAKE"}

        ranked=self.ranker.rank(claims[0],evidence)

        results=[]

        for claim in claims[:3]:

            verdict,conf,exp=self.verifier.verify(claim,ranked)

            results.append(verdict)

        fake_count=results.count("FAKE")
        real_count=results.count("REAL")

        final="FAKE" if fake_count>real_count else "REAL"

        return {"verdict":final}

In [3]:
import pandas as pd
fake_df = pd.read_csv("/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/Fake.csv")
true_df = pd.read_csv("/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/True.csv")

# assign labels
fake_df["label"] = 0   # FAKE
true_df["label"] = 1   # REAL

# combine datasets
df = pd.concat([fake_df, true_df], axis=0)

# shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# keep only needed columns
df = df[["text","label"]]

print("Dataset loaded:", df.shape)
df.head()

Dataset loaded: (44898, 2)


,text,label
0,"21st Century Wire says Ben Stein, reputable pr...",0
1,WASHINGTON (Reuters) - U.S. President Donald T...,1
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,1
3,"On Monday, Donald Trump once again embarrassed...",0
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",1


In [4]:
pipeline = FakeNewsPipeline()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:


y_true=[]
y_pred=[]

data = df.sample(100,random_state=42)

for _,row in data.iterrows():

    article=row["text"]

    label="REAL" if row["label"]==1 else "FAKE"

    result=pipeline.detect(article)

    predicted=result["verdict"]

    y_true.append(label)
    y_pred.append(predicted)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

In [6]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report

accuracy=accuracy_score(y_true,y_pred)

precision=precision_score(y_true,y_pred,pos_label="REAL")

recall=recall_score(y_true,y_pred,pos_label="REAL")

f1=f1_score(y_true,y_pred,pos_label="REAL")

print("\n==============================")
print("MODEL EVALUATION")
print("==============================")

print("Accuracy:",round(accuracy,2))
print("Precision:",round(precision,2))
print("Recall:",round(recall,2))
print("F1 Score:",round(f1,2))

print("\nDetailed Report:\n")
print(classification_report(y_true,y_pred))


MODEL EVALUATION
Accuracy: 0.59
Precision: 0.53
Recall: 0.76
F1 Score: 0.62

Detailed Report:

              precision    recall  f1-score   support

        FAKE       0.69      0.45      0.55        55
        REAL       0.53      0.76      0.62        45

    accuracy                           0.59       100
   macro avg       0.61      0.61      0.59       100
weighted avg       0.62      0.59      0.58       100



# liar

In [10]:
import pandas as pd

liar_df = pd.read_csv(
    "/kaggle/input/datasets/doanquanvietnamca/liar-dataset/train.tsv",
    sep="\t",
    header=None
)

# rename important columns
liar_df.columns = [
    "id","label","statement","subject","speaker",
    "speaker_job","state","party","barely_true",
    "false","half_true","mostly_true","pants_fire","context"
]

# keep only needed columns
liar_df = liar_df[["statement","label"]]

liar_df.head()

,statement,label
0,Says the Annies List political group supports ...,false
1,When did the decline of coal start? It started...,half-true
2,"Hillary Clinton agrees with John McCain ""by vo...",mostly-true
3,Health care reform legislation is likely to ma...,false
4,The economic turnaround started at the end of ...,half-true


In [11]:
liar_df["label"] = liar_df["label"].apply(
    lambda x: "FAKE" if x in ["false","pants-fire","barely-true"] else "REAL"
)

In [14]:
pipeline = FakeNewsPipeline()

y_true=[]
y_pred=[]

fake_samples = liar_df[liar_df["label"]=="FAKE"].sample(200, random_state=42)
real_samples = liar_df[liar_df["label"]=="REAL"].sample(200, random_state=42)

data = pd.concat([fake_samples, real_samples])

for _,row in data.iterrows():

    claim=row["statement"]
    label=row["label"]

    result=pipeline.detect(claim)

    predicted=result["verdict"]

    y_true.append(label)
    y_pred.append(predicted)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

In [15]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report

accuracy=accuracy_score(y_true,y_pred)

precision=precision_score(y_true,y_pred,pos_label="REAL")

recall=recall_score(y_true,y_pred,pos_label="REAL")

f1=f1_score(y_true,y_pred,pos_label="REAL")

print("\n==============================")
print("MODEL EVALUATION")
print("==============================")

print("Accuracy:",round(accuracy,2))
print("Precision:",round(precision,2))
print("Recall:",round(recall,2))
print("F1 Score:",round(f1,2))

print("\nDetailed Report:\n")
print(classification_report(y_true,y_pred))


MODEL EVALUATION
Accuracy: 0.5
Precision: 0.5
Recall: 0.61
F1 Score: 0.55

Detailed Report:

              precision    recall  f1-score   support

        FAKE       0.50      0.38      0.43       200
        REAL       0.50      0.61      0.55       200

    accuracy                           0.50       400
   macro avg       0.50      0.50      0.49       400
weighted avg       0.50      0.50      0.49       400



# welfake dataset

In [5]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/nitaisatapathy/welfake-dataset/WELFake_Dataset.csv")

df = df[["text","label"]]

# convert label
df["label"] = df["label"].apply(lambda x: "REAL" if x==1 else "FAKE")

df = df.dropna().reset_index(drop=True)

In [7]:
fake_samples = df[df["label"]=="FAKE"].sample(150, random_state=42)
real_samples = df[df["label"]=="REAL"].sample(150, random_state=42)

data = pd.concat([fake_samples, real_samples])

In [11]:
pipeline = FakeNewsPipeline()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [12]:
y_true=[]
y_pred=[]

for _,row in data.iterrows():

    article=row["text"]
    label=row["label"]

    result=pipeline.detect(article)

    predicted=result["verdict"]

    y_true.append(label)
    y_pred.append(predicted)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

In [15]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report

accuracy=accuracy_score(y_true,y_pred)
precision=precision_score(y_true,y_pred,pos_label="REAL")
recall=recall_score(y_true,y_pred,pos_label="REAL")
f1=f1_score(y_true,y_pred,pos_label="REAL")

print("\n==============================")
print("MODEL EVALUATION")
print("==============================")

print("Accuracy:",round(accuracy,2))
print("Precision:",round(precision,2))
print("Recall:",round(recall,2))
print("F1 Score:",round(f1,2))

print("\nDetailed Report:\n")
print(classification_report(y_true,y_pred))


MODEL EVALUATION
Accuracy: 0.34
Precision: 0.36
Recall: 0.41
F1 Score: 0.38

Detailed Report:

              precision    recall  f1-score   support

        FAKE       0.31      0.27      0.29       150
        REAL       0.36      0.41      0.38       150

    accuracy                           0.34       300
   macro avg       0.33      0.34      0.33       300
weighted avg       0.33      0.34      0.33       300

